## Neteja, normalitzacio, centralitzacio

In [11]:
import pandas as pd
import numpy as np
from datetime import timedelta

# ==========================================
# 1. CARGA DE DATOS (Rutas relativas a /Data)
# ==========================================
path = "../Data/"
ventas = pd.read_csv(f"{path}Ventas.csv")
productos = pd.read_csv(f"{path}Productos.csv")
clientes = pd.read_csv(f"{path}Clientes.csv")
campanas = pd.read_csv(f"{path}Campañas.csv")
potencial = pd.read_csv(f"{path}Potencial.csv")

# Formatear fechas
ventas['Fecha'] = pd.to_datetime(ventas['Fecha'])
campanas['Fecha inicio'] = pd.to_datetime(campanas['Fecha inicio'])
campanas['Fecha fin'] = pd.to_datetime(campanas['Fecha fin'])

# ==========================================
# 2. CAPA 2: LIMPIEZA DE RUIDO ADMINISTRATIVO
# ==========================================
ventas['abs_valor'] = ventas['Valores_H'].abs()
duplicados_admin = ventas.duplicated(subset=['Fecha', 'Id. Cliente', 'Id. Producto', 'abs_valor'], keep=False)
ventas_sin_ruido = ventas[~duplicados_admin].copy()

# ==========================================
# 3. UNIFICACIÓN Y AGREGACIÓN (CORREGIDO)
# ==========================================
# Incluimos todas las columnas necesarias en el merge y en el groupby
df = ventas_sin_ruido.merge(productos[['Id.Prod', 'Familia_H',
'Categoria_H', 'Bloque analítico']],
                            left_on='Id. Producto', right_on='Id.Prod')
df = df.merge(clientes[['Id. Cliente', 'Provincia']], on='Id. Cliente')

# Agregamos ventas. NOTA: Incluimos 'Bloque analítico' y 'Categoria_H' en el groupby
df_daily = df.groupby(['Id. Cliente', 'Provincia', 'Categoria_H',
'Familia_H', 'Bloque analítico', 'Fecha']).agg({
    'Valores_H': 'sum',
    'Unidades': 'sum'
}).reset_index()

# ==========================================
# 4. DETECCIÓN DE PERIODOS PROMOCIONALES
# ==========================================
def marcar_promos(fecha):
    mask = (campanas['Fecha inicio'] <= fecha) & (campanas['Fecha fin']
>= fecha)
    return 1 if mask.any() else 0

df_daily['is_promo_period'] = df_daily['Fecha'].apply(marcar_promos)

# ==========================================
# 5. NORMALIZACIÓN VS POTENCIAL (REPARADO)
# ==========================================
# Aseguramos tipos de datos para el merge
df_daily['Id. Cliente'] = df_daily['Id. Cliente'].astype(int)
df_daily['Categoria_H'] = df_daily['Categoria_H'].astype(str).str.strip()
potencial['Id.Cliente'] = potencial['Id.Cliente'].astype(int)
potencial['Categoria Productos'] = potencial['Categoria Productos'].astype(str).str.strip()

# Agrupamos potencial por cliente y categoría (vínculo real entrearchivos)
pot_grouped = potencial.groupby(['Id.Cliente', 'Categoria Productos'])['Potencial'].sum().reset_index()

# Merge por cliente y categoría
df_daily = df_daily.merge(pot_grouped,
                            left_on=['Id. Cliente', 'Categoria_H'],
                            right_on=['Id.Cliente', 'Categoria Productos'],
                            how='left')

# Cálculo del Ratio
df_daily['ratio_vs_potential'] = np.where(df_daily['Potencial'] > 0,
                                            df_daily['Valores_H'] /
df_daily['Potencial'],
                                            0)

# ==========================================
# 6. IDENTIFICACIÓN DE OUTLIERS
# ==========================================
def flag_outliers(series):
    if len(series) < 5: return pd.Series(0, index=series.index)
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    limit = Q3 + (1.5 * (Q3 - Q1))
    return (series > limit).astype(int)

df_daily['evento_especial'] = df_daily.groupby(['Id. Cliente',
'Familia_H'])['Valores_H'].transform(flag_outliers)

# ==========================================
# 7. SEGMENTACIÓN POR ANTIGÜEDAD
# ==========================================
primer_pedido = df_daily.groupby('Id. Cliente')['Fecha'].transform('min')
df_daily['historico_incompleto'] = ((df_daily['Fecha'] -
primer_pedido).dt.days < 180).astype(int)

# ==========================================
# 8. GUARDADO DE RESULTADO (FINAL)
# ==========================================
cols_finales = [
    'Fecha', 'Id. Cliente', 'Provincia', 'Familia_H', 'Bloque analítico',
    'Valores_H', 'ratio_vs_potential', 'is_promo_period',
    'evento_especial', 'historico_incompleto'
]

df_final = df_daily[cols_finales].sort_values(['Id. Cliente', 'Fecha'])
df_final.to_csv(f"{path}Dataset_Limpiado_Normalizado.csv", index=False)

# Verificaciones finales
print("✅ Proceso completado.")
print(f"Ratios positivos calculados: {(df_final['ratio_vs_potential'] > 0).sum()}")
print(f"Columnas en el archivo final: {df_final.columns.tolist()}")

C:\Users\Albert\AppData\Local\Temp\ipykernel_4280\1949020166.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  ventas = pd.read_csv(f"{path}Ventas.csv")


✅ Proceso completado.
Ratios positivos calculados: 63750
Columnas en el archivo final: ['Fecha', 'Id. Cliente', 'Provincia', 'Familia_H', 'Bloque analítico', 'Valores_H', 'ratio_vs_potential', 'is_promo_period', 'evento_especial', 'historico_incompleto']


In [12]:
# ==========================================================
# TEST DE DIAGNÓSTICO (Ejecuta esto para ver qué falla)
# ==========================================================
nulos = df_daily['Potencial'].isna().sum()
if nulos > 0:
    print(f"⚠️ ATENCIÓN: {nulos} filas no encontraron su potencial.")
    print("Muestra de categorías en Ventas:", df_daily['Bloque analítico'].unique()[:3])
    print("Muestra de categorías en Potencial:", potencial['Categoria Productos'].unique()[:3])
else:
    print("✅ Potencial cruzado correctamente.")

⚠️ ATENCIÓN: 52579 filas no encontraron su potencial.
Muestra de categorías en Ventas: ['Commodities' 'Productos Técnicos']
Muestra de categorías en Potencial: ['Categoria C1' 'Categoria C2' 'Categoria T1']


In [13]:
df_final["ratio_vs_potential"].mean()

0.11855407590286851